# Label smoothing on MNIST 3 vs 7
This notebook installs and calls the tested project package; experiment logic stays in `src/`. Connect it to a Colab GPU kernel before running. See `README.md` for the complete VS Code workflow.

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'Connect this notebook to a Colab GPU server'
!nvidia-smi

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

project_dir = Path('/content/LS-tiny-project')
repository_url = 'https://github.com/avkornaev/LS-tiny-project.git'
if (project_dir / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(project_dir), 'pull', '--ff-only'], check=True
    )
elif project_dir.exists():
    raise RuntimeError(
        f'{project_dir} exists but is not a Git checkout; remove it first'
    )
else:
    subprocess.run(
        ['git', 'clone', repository_url, str(project_dir)], check=True
    )
os.chdir(project_dir)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[dev]'],
    check=True,
)
print('Installed project from', project_dir)

In [ ]:
!pytest -q
!ruff check .

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
output_dir = '/content/drive/MyDrive/LS-tiny-project/reports'

In [ ]:
# This is the full six-run experiment and may take several minutes.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        '-m',
        'label_smoothing.experiment',
        '--all',
        '--device',
        'cuda',
        '--output-dir',
        output_dir,
    ],
    check=True,
)

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

completed_runs = list(Path(output_dir).glob('*_all'))
assert completed_runs, (
    'No timestamped full run exists. Run the training cell successfully first.'
)
run_dir = max(completed_runs, key=lambda path: path.stat().st_mtime)
results_dir = run_dir / 'results'
runs_path = results_dir / 'runs.csv'
summary_path = results_dir / 'summary.csv'
assert runs_path.is_file(), 'Run the full training cell successfully first'
assert summary_path.is_file(), 'The full six-run experiment did not finish'
display(pd.read_csv(runs_path))
display(pd.read_csv(summary_path))
figure_paths = [
    run_dir / 'figures/reliability.png',
    run_dir / 'figures/confidence.png',
    run_dir / 'figures/validation_loss.png',
    run_dir / 'figures/tsne.png',
]
for path in figure_paths:
    display(Image(filename=path))